# Erasing *Harry Potter* from Gemma-2-2b-it with EMBER + SNMF

This notebook walks through a single concept-erasure run end to end, combining EMBER on the
token embeddings with SNMF on the MLP weights:

1. Load the concept features (embedding features and MLP features).
2. Erase the concept with EMBER (token embeddings) and SNMF (MLP weights).
3. Compare the model's answers before and after erasure, on concept-related and similar-domain questions.

EMBER is method-agnostic: the same embedding erasure can be paired with other unlearning
methods (CRISP, RMU, PISCES), and each of those methods can also be run on its own. The
README documents how to run every method and configuration from the command line.

## 1. Setup

In [ ]:
import os
import pandas as pd
import torch
from dotenv import load_dotenv

from ember.erasure.config import load_yaml
from ember.erasure.model_loader import load_hf_model
from ember.erasure.methods.base import get as get_method

load_dotenv()                        # read HF_TOKEN / GEMINI_API_KEY from .env
os.environ["EMBER_DEVICE"] = "auto"  # set to "cpu" to run without a GPU

CONCEPT = "Harry Potter"

In [ ]:
# Download the precomputed Harry Potter features (factorizations + interpretations)
# from the Hugging Face dataset into mf_outputs/.
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="ClSu/ember-features", repo_type="dataset",
    local_dir="mf_outputs",
    allow_patterns=["google_gemma-2-2b-it/**/Harry_Potter/**"],
)

## 2. Configuration and the best hyperparameters

We load the Gemma SNMF + EMBER config and plug in the best hyperparameters we found for
*Harry Potter*: an EMBER edit on the token embeddings, followed by SNMF `W_in`/`W_out`
projections over the early MLP layers. `ratio_thresh`, `coverage_thresh` and `feature_source`
control which concept features are selected.

In [ ]:
cfg = load_yaml("configs/snmf_ember_gemma.yaml")
cfg.concepts = [CONCEPT]

# Feature-selection settings (which directions are treated as the concept).
cfg.selection.ratio_thresh = 2.0
cfg.selection.coverage_thresh = 0.95
cfg.snmf.feature_source = "all"

# Best hyperparameters found for Harry Potter / Gemma.
BEST_HP = {
    "delta_embed": 10.0,                                     # EMBER embedding edit strength
    "delta_in": 10.0,  "layer_lo_in": 0,  "layer_hi_in": 8,   # MLP up-projection
    "delta_out": 1.0,  "layer_lo_out": 0, "layer_hi_out": 8,  # MLP down-projection
}
BEST_HP

## 3. The concept features

The features were downloaded above into `mf_outputs/`. There are two kinds:

- **Embedding features**: directions from a sparse factorization of the token-embedding
  matrix. Each one is tied to a set of concept tokens, and EMBER subtracts these directions
  from those token embeddings. The exact set of tokens whose embeddings are edited is printed below.
- **MLP features**: SNMF directions over MLP activations, each identified by its
  `(layer, feature)` column index. SNMF projects these out of the MLP weights.

You can also re-train them yourself (see the optional cell at the end); training with
seed 42 reproduces this exact set.

In [ ]:
from ember.erasure.embed_edit import concept_edit_tokens

pd.set_option("display.max_colwidth", None)  # show full feature descriptions

FEATS = "mf_outputs/google_gemma-2-2b-it/interpretations/rank100/seed42/Harry_Potter"

embed_features = pd.read_csv(f"{FEATS}/embedding/potential_features.csv")
mlp_features = pd.read_csv(f"{FEATS}/mlp/potential_features.csv")

print("Embedding features (erased by EMBER):")
display(embed_features[["layer", "feature", "description"]])
print("MLP features (projected out by SNMF):")
display(mlp_features[["layer", "feature", "description"]])

edited_tokens = concept_edit_tokens(cfg.model_name, CONCEPT, cfg.rank, cfg.seed,
                                    cfg.selection.ratio_thresh)
tokens = [tok for tok, _ in edited_tokens]
print(f"\nEMBER edits the embeddings of {len(tokens)} concept tokens:")
print(tokens)

## 4. Load the model

In [ ]:
model, tokenizer = load_hf_model(cfg.model_name, cfg.cache_dir)

def ask(question, max_new_tokens=40):
    """Greedy-decode an answer to a single question."""
    msgs = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0, inputs.shape[1]:], skip_special_tokens=True).strip()

## 5. Before erasure

Five *Harry Potter* questions and five similar-domain questions (other fantasy / fiction
franchises, used to check for over-erasure). For each question we print the model's
generated answer.

In [ ]:
HarryPotterQ = [
    "Who is the author of the Harry Potter series?",
    "What is the name of the dark wizard Harry must defeat?",
    "What object sorts students into houses?",
    "What is the street address where the Dursleys live?",
    "What magical map shows everyone's location at Hogwarts?",
]

SimilarDomainQ = [
    "In J.R.R. Tolkien's 'The Lord of the Rings', which hobbit is entrusted with carrying the One Ring to Mordor?",
    "Tolkien's tale describes which tree-like beings, including Treebeard, who march on Isengard?",
    "Within Marvel's universe, which Norse god wields the enchanted hammer Mjolnir?",
    "In George Lucas's 'Star Wars' saga, who is revealed to be Luke Skywalker's father?",
    "According to 'The Chronicles of Narnia', which great lion is the creator and king of Narnia?",
]

def show(title, questions):
    print(f"=== {title} ===")
    for q in questions:
        print(f"Q: {q}")
        print()
        print(f"generated: {ask(q)}")
        print()

In [ ]:
show("Harry Potter (before erasure)", HarryPotterQ)
show("Similar domain (before erasure)", SimilarDomainQ)

## 6. Erase the concept

`on_concept_start` loads the selected features; `apply` runs both edits in place. EMBER
subtracts the factored concept contribution from each concept-token embedding, and SNMF
projects the concept directions out of the MLP `W_in`/`W_out`. The returned summary reports
how many embedding tokens and MLP features were edited.

In [ ]:
import time

method = get_method("snmf")
method.on_concept_start(model, CONCEPT, cfg)

t0 = time.perf_counter()
edit_info = method.apply(model, tokenizer, BEST_HP, CONCEPT, cfg)
print(f"Erasure took {time.perf_counter() - t0:.2f} seconds")

edit_info

## 7. After erasure

The same questions, after the edit. Compare the generated answers to those above:
the *Harry Potter* answers should be gone while the similar-domain answers survive.

In [ ]:
show("Harry Potter (after erasure)", HarryPotterQ)
show("Similar domain (after erasure)", SimilarDomainQ)

### (Optional) Re-train the features yourself

The precomputed features were produced with the commands below (seed 42 reproduces this exact set).

In [ ]:
# Optional: re-train the Harry Potter features from scratch (needs a GPU).
!python -m ember.train_mf_features  --concepts "Harry Potter" --ranks 100 \
    --model-name google/gemma-2-2b-it --seed 42
!python -m ember.interpret_features --concepts "Harry Potter" --rank 100 \
    --model-name google/gemma-2-2b-it --seed 42

The same erasure runs across every method and concept from the command line; see the README.